# Câncer de Mama — Modelos de Machine Learning

Este notebook treina e avalia dois modelos de classificação para prever o desfecho (`Status`: Alive / Dead) de pacientes com câncer de mama.

**Modelos escolhidos:**
- **Regressão Logística** — baseline interpretável, adequada para classificação binária com features codificadas
- **Random Forest** — robusto a outliers e dados mistos, suporta `class_weight='balanced'` para lidar com desbalanceamento, e fornece importância de features nativamente

**Justificativa da escolha:**
O dataset possui 4.024 amostras com 11 features categóricas e 4 numéricas. Há desbalanceamento de classe (~85% Alive / ~15% Dead). 
Regressão Logística e Random Forest complementam-se: um é simples e interpretável, o outro é não-linear e mais robusto. 
Ambos suportam SHAP e métricas voltadas para a classe minoritária (recall/F1), críticas em diagnóstico médico.

## 1. Importações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay, f1_score, recall_score
)

import shap
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
print("Imports OK")

## 2. Carregamento dos Dados

In [ ]:
df = pd.read_csv('base/Breast_Cancer.csv')
df.columns = df.columns.str.strip()
df['Marital Status'] = df['Marital Status'].str.strip()

print(f"Shape: {df.shape}")
print(f"\nDistribuição do alvo:\n{df['Status'].value_counts()}")
print(f"\nProporção:\n{df['Status'].value_counts(normalize=True).round(3)}")
df.head()

## 3. Pré-processamento

### 3.1 Encode do alvo e separação de features

In [ ]:
# Encode alvo: Alive=0, Dead=1
le = LabelEncoder()
y = le.fit_transform(df['Status'])   # Dead=1 é a classe positiva (risco)
print("Classes:", le.classes_, "→ índices:", list(range(len(le.classes_))))

# 'Survival Months' é removida: trata-se de dado prospectivo (quanto tempo
# o paciente viveu APÓS o diagnóstico). No momento do diagnóstico essa
# informação não existe — mantê-la causaria data leakage.
X = df.drop(columns=['Status', 'Survival Months'])
print("⚠️  'Survival Months' removida para evitar data leakage.")

# Separar colunas numéricas e categóricas
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f"\nNuméricas ({len(num_cols)}): {num_cols}")
print(f"Categóricas ({len(cat_cols)}): {cat_cols}")

### 3.2 Pipeline de pré-processamento

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Encoder genérico para variáveis nominais/ordinais
# handle_unknown='use_encoded_value' evita erros no test set
cat_transformer = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
num_transformer = StandardScaler()

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
], remainder='drop')

print("Preprocessor criado.")

### 3.3 Divisão treino / teste (80/20 estratificado)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"Dead no treino: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Dead no teste:  {y_test.sum()} ({y_test.mean():.1%})")

## 4. Função de Avaliação

In [ ]:
def evaluate_model(name, pipeline, X_tr, y_tr, X_te, y_te):
    """Treina o pipeline e retorna métricas completas."""
    pipeline.fit(X_tr, y_tr)
    y_pred  = pipeline.predict(X_te)
    y_proba = pipeline.predict_proba(X_te)[:, 1]

    print(f"{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(classification_report(y_te, y_pred, target_names=le.classes_))
    print(f"ROC-AUC : {roc_auc_score(y_te, y_proba):.4f}")

    # Cross-validation no conjunto de treino
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_f1 = cross_val_score(pipeline, X_tr, y_tr, cv=cv, scoring='f1')
    print(f"CV F1 (5-fold, treino): {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")

    return pipeline, y_pred, y_proba

print("Função de avaliação definida.")

## 5. Modelo 1 — Regressão Logística

**Por quê?** Modelo linear interpretável, eficiente para classificação binária.  
`class_weight='balanced'` compensa o desbalanceamento (~85/15).  
`max_iter=1000` garante convergência com features escaladas.

In [ ]:
lr_pipeline = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
        solver='lbfgs'
    ))
])

lr_pipe, lr_pred, lr_proba = evaluate_model(
    "Regressão Logística", lr_pipeline, X_train, y_train, X_test, y_test
)

## 6. Modelo 2 — Random Forest

**Por quê?** Conjunto de árvores de decisão, não-linear, robusto a outliers e dados mistos.  
`class_weight='balanced_subsample'` aplica balanceamento em cada árvore.  
`n_estimators=300` oferece boa estabilidade sem custo excessivo.

In [ ]:
rf_pipeline = Pipeline([
    ('pre', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=300,
        class_weight='balanced_subsample',
        max_depth=None,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipe, rf_pred, rf_proba = evaluate_model(
    "Random Forest", rf_pipeline, X_train, y_train, X_test, y_test
)

## 7. Comparação Visual dos Modelos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Confusion matrices ---
for ax, pred, name in zip(axes[:2],
                           [lr_pred, rf_pred],
                           ['Regressão Logística', 'Random Forest']):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Matriz de Confusão\n{name}')

# --- ROC Curves ---
ax = axes[2]
for proba, name, color in zip([lr_proba, rf_proba],
                                ['Regressão Logística', 'Random Forest'],
                                ['royalblue', 'forestgreen']):
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, lw=2)

ax.plot([0,1],[0,1],'k--', lw=1, label='Aleatório')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curva ROC')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.suptitle('Comparação de Modelos', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 7.1 Tabela-Resumo de Métricas

In [ ]:
from sklearn.metrics import accuracy_score, precision_score

rows = []
for name, pred, proba in [
    ('Regressão Logística', lr_pred, lr_proba),
    ('Random Forest',       rf_pred, rf_proba)
]:
    rows.append({
        'Modelo':    name,
        'Accuracy':  accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall':    recall_score(y_test, pred),
        'F1-Score':  f1_score(y_test, pred),
        'ROC-AUC':   roc_auc_score(y_test, proba)
    })

metrics_df = pd.DataFrame(rows).set_index('Modelo').round(4)
display(metrics_df.style.highlight_max(axis=0, color='#d4edda').format('{:.4f}'))

## 8. Importância de Features

### 8.1 Random Forest — Feature Importance (MDI)

In [ ]:
feature_names = num_cols + cat_cols

importances = rf_pipe.named_steps['clf'].feature_importances_
feat_df = (pd.DataFrame({'Feature': feature_names, 'Importance': importances})
           .sort_values('Importance', ascending=True))

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(feat_df['Feature'], feat_df['Importance'], color='steelblue', edgecolor='white')
ax.set_xlabel('Importância (MDI)')
ax.set_title('Random Forest — Importância das Features')
plt.tight_layout()
plt.show()

### 8.2 Regressão Logística — Coeficientes

In [ ]:
coef = lr_pipe.named_steps['clf'].coef_[0]
coef_df = (pd.DataFrame({'Feature': feature_names, 'Coefficient': coef})
           .sort_values('Coefficient', ascending=True))

colors = ['tomato' if c < 0 else 'steelblue' for c in coef_df['Coefficient']]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Coeficiente (log-odds)')
ax.set_title('Regressão Logística — Coeficientes\n(positivo → maior risco de Dead; negativo → menor risco)')
plt.tight_layout()
plt.show()

## 9. Explicabilidade — SHAP

SHAP (SHapley Additive exPlanations) quantifica a contribuição de cada feature para cada predição individual.

### 9.1 Random Forest — SHAP Summary Plot

In [ ]:
# Transformar X_test com o preprocessor
X_test_transformed = rf_pipe.named_steps['pre'].transform(X_test)
X_test_df = pd.DataFrame(X_test_transformed, columns=feature_names)

explainer_rf = shap.TreeExplainer(rf_pipe.named_steps['clf'])
shap_values_rf = explainer_rf.shap_values(X_test_df)

# SHAP >= 0.46: shap_values retorna array 3D (n_samples, n_features, n_classes)
# Extraímos a fatia da classe Dead (índice 1): shape -> (n_samples, n_features)
if isinstance(shap_values_rf, list):
    sv_dead = shap_values_rf[1]          # API antiga
else:
    sv_dead = shap_values_rf[:, :, 1]    # API nova (3D)

shap.summary_plot(
    sv_dead, X_test_df,
    plot_type='bar', show=False,
    plot_size=(9, 5)
)
plt.title('SHAP — Random Forest (classe Dead)', fontsize=13)
plt.tight_layout()
plt.show()

### 9.2 Random Forest — SHAP Beeswarm (detalhe por amostra)

In [ ]:
shap.summary_plot(
    sv_dead, X_test_df,
    show=False, plot_size=(9, 5)
)
plt.title('SHAP Beeswarm — Random Forest (classe Dead)', fontsize=13)
plt.tight_layout()
plt.show()

### 9.3 Regressão Logística — SHAP Summary Plot

In [ ]:
X_test_lr = lr_pipe.named_steps['pre'].transform(X_test)
X_test_lr_df = pd.DataFrame(X_test_lr, columns=feature_names)

# LinearExplainer para modelos lineares
# masker com dados de background para a API nova do SHAP
masker = shap.maskers.Independent(X_test_lr_df, max_samples=100)
explainer_lr = shap.LinearExplainer(
    lr_pipe.named_steps['clf'],
    masker=masker
)
shap_values_lr = explainer_lr.shap_values(X_test_lr_df)

# Para classificação binária, LinearExplainer retorna array 2D diretamente
shap.summary_plot(
    shap_values_lr, X_test_lr_df,
    plot_type='bar', show=False,
    plot_size=(9, 5)
)
plt.title('SHAP — Regressão Logística (classe Dead)', fontsize=13)
plt.tight_layout()
plt.show()

## 10. Discussão dos Resultados

### Métricas relevantes para o problema

Em diagnóstico oncológico, **recall da classe Dead** é a métrica mais crítica:  
- Um **falso negativo** (predizer Alive quando o paciente vai a óbito) é muito mais grave do que um falso positivo.  
- Por isso, priorizamos **Recall** e **F1-Score** sobre Accuracy.

### Qual modelo usar?

| Cenário | Modelo recomendado |
|---|---|
| Triagem de alto risco (priorizar recall) | Random Forest com limiar ajustado |
| Explicabilidade para médico | Regressão Logística (coeficientes diretos) |
| Produção com dados estruturados | Random Forest (mais robusto) |

### Limitações

1. **Desbalanceamento**: ~85% Alive — ambos os modelos usam `class_weight='balanced'`, mas o recall da classe Dead ainda pode ser limitado sem técnicas como SMOTE.  
2. **`Survival Months` removida**: essa coluna representa o tempo de sobrevida do paciente após o diagnóstico — uma informação prospectiva que não existe no momento da consulta. Mantê-la causaria **data leakage** severo (o modelo "aprende" diretamente quem morreu, sem usar sinais clínicos reais).  
3. **O médico tem a palavra final**: estes modelos são ferramentas de **suporte ao diagnóstico**, não substitutos da decisão clínica. Os resultados devem ser sempre revisados por um profissional de saúde.

### Próximos passos sugeridos

- Testar SMOTE para oversampling da classe Dead
- Ajustar limiar de decisão (threshold tuning) para maximizar recall
- Avaliar XGBoost como alternativa ao Random Forest